# Hydrogen Permeation Models with Surface Chemistry

This notebook implements comprehensive hydrogen permeation models combining multiple physics layers:

- **L2**: Perfect Oxide diffusion
- **L3**: Defective Oxide (parallel paths: intact, pinhole, crack, grain boundary)
- **L4**: Defective Metal (microstructure effects: GB enhancement, trapping)
- **L6**: Surface Kinetics (dissociation/recombination at gas-oxide interface)

The models solve coupled steady-state flux equations:
$$J_{surface} = J_{oxide} = J_{metal}$$

## 1. Import Required Libraries and Data Modules

In [1]:
import numpy as np
from scipy.optimize import brentq
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact
import pandas as pd
from itertools import groupby
from operator import itemgetter

# Import data dictionaries
from data.surface_kinetics_data import SURFACE_KINETICS, get_surface_kinetics
from data.oxide_properties import OXIDE_PROPERTIES
from data.material_data import MATERIALS

# Gas constant
R = 8.314  # J/(mol·K)

## 2. Define Core Flux Functions (L2+L6)

### Mathematical Framework

**Surface Dissociation Flux:**
$$J_{surface} = k_{diss} \cdot P_{up} \cdot (1 - \theta)^2 - k_{recomb} \cdot \theta^2$$

**Oxide Diffusion Flux:**
$$J_{oxide} = \alpha \cdot \left( g(\theta) - \sqrt{P_{int}} \right)$$

**Metal Diffusion Flux:**
$$J_{metal} = \beta \cdot \left( \sqrt{P_{int}} - \sqrt{P_{down}} \right)$$

Where:
- $\alpha = \frac{D_{ox} \cdot K_{ox}}{L_{ox}}$ (oxide permeance)
- $\beta = \frac{D_m \cdot K_{s,m}}{L_m}$ (metal permeance)
- $g(\theta) = \frac{\theta}{(1 - \theta) \sqrt{K_{eq}}}$ (concentration function)

In [2]:
def g_theta(theta, K_eq):
    """
    Concentration function: g(θ) = θ / ((1-θ) × √K_eq)
    
    This converts surface coverage to effective √P at oxide surface.
    
    Parameters
    ----------
    theta : float
        Surface coverage (0 to 1)
    K_eq : float
        Surface equilibrium constant [Pa⁻¹]
    
    Returns
    -------
    float
        Effective √P value
    """
    if theta >= 1.0:
        return np.inf
    return theta / ((1.0 - theta) * np.sqrt(K_eq))


def sqrt_P_int_from_theta(theta, alpha, beta, K_eq, P_down):
    """
    Solve for √P_int analytically from flux balance (J_oxide = J_metal).
    
    √P_int = (α × g(θ) + β × √P_down) / (α + β)
    
    Parameters
    ----------
    theta : float
        Surface coverage
    alpha : float
        Oxide permeance [mol/m²/s/Pa^0.5]
    beta : float
        Metal permeance [mol/m²/s/Pa^0.5]
    K_eq : float
        Surface equilibrium constant [Pa⁻¹]
    P_down : float
        Downstream pressure [Pa]
    
    Returns
    -------
    float
        √P_int value
    """
    g = g_theta(theta, K_eq)
    sqrt_P_down = np.sqrt(P_down)
    return (alpha * g + beta * sqrt_P_down) / (alpha + beta)


def surface_flux(theta, P_up, k_diss, K_eq):
    """
    Calculate surface dissociation flux.
    
    J_surface = k_diss × P_up × (1-θ)² - k_recomb × θ²
    
    Parameters
    ----------
    theta : float
        Surface coverage
    P_up : float
        Upstream pressure [Pa]
    k_diss : float
        Dissociation rate constant [mol/m²/s/Pa]
    K_eq : float
        Equilibrium constant [Pa⁻¹]
    
    Returns
    -------
    float
        Surface flux [mol/m²/s]
    """
    k_recomb = k_diss / K_eq
    J_surface = k_diss * P_up * (1 - theta)**2 - k_recomb * theta**2
    return J_surface


def oxide_flux(theta, alpha, beta, K_eq, P_down):
    """
    Calculate oxide diffusion flux.
    
    J_oxide = α × (g(θ) - √P_int)
    
    Parameters
    ----------
    theta : float
        Surface coverage
    alpha : float
        Oxide permeance [mol/m²/s/Pa^0.5]
    beta : float
        Metal permeance [mol/m²/s/Pa^0.5]
    K_eq : float
        Surface equilibrium constant [Pa⁻¹]
    P_down : float
        Downstream pressure [Pa]
    
    Returns
    -------
    float
        Oxide flux [mol/m²/s]
    """
    g = g_theta(theta, K_eq)
    sqrt_P_int = sqrt_P_int_from_theta(theta, alpha, beta, K_eq, P_down)
    J_oxide = alpha * (g - sqrt_P_int)
    return J_oxide


def metal_flux(theta, alpha, beta, K_eq, P_down):
    """
    Calculate metal diffusion flux.
    
    J_metal = β × (√P_int - √P_down)
    
    Parameters
    ----------
    theta : float
        Surface coverage
    alpha : float
        Oxide permeance [mol/m²/s/Pa^0.5]
    beta : float
        Metal permeance [mol/m²/s/Pa^0.5]
    K_eq : float
        Surface equilibrium constant [Pa⁻¹]
    P_down : float
        Downstream pressure [Pa]
    
    Returns
    -------
    float
        Metal flux [mol/m²/s]
    """
    sqrt_P_int = sqrt_P_int_from_theta(theta, alpha, beta, K_eq, P_down)
    sqrt_P_down = np.sqrt(P_down)
    J_metal = beta * (sqrt_P_int - sqrt_P_down)
    return J_metal


def surface_flux_residual(theta, P_up, alpha, beta, P_down, k_diss, K_eq):
    """
    Residual for root-finding: J_surface - J_oxide = 0
    
    We substitute √P_int(θ) into J_oxide and solve for θ.
    
    Parameters
    ----------
    theta : float
        Surface coverage (variable to solve for)
    P_up : float
        Upstream pressure [Pa]
    alpha : float
        Oxide permeance [mol/m²/s/Pa^0.5]
    beta : float
        Metal permeance [mol/m²/s/Pa^0.5]
    P_down : float
        Downstream pressure [Pa]
    k_diss : float
        Dissociation rate constant [mol/m²/s/Pa]
    K_eq : float
        Equilibrium constant [Pa⁻¹]
    
    Returns
    -------
    float
        Residual value (should be zero at solution)
    """
    J_surface = surface_flux(theta, P_up, k_diss, K_eq)
    J_oxide = oxide_flux(theta, alpha, beta, K_eq, P_down)
    return J_surface - J_oxide

## 3. Implement Perfect Oxide + Perfect Metal Solver (L2+L6)

This solver finds the steady-state solution where all three fluxes are equal:
$$J_{surface} = J_{oxide} = J_{metal}$$

It also performs rate-limiting analysis based on resistance fractions.

In [3]:
def solve_steady_state_flux_direct(P_up, P_down, L_m, k_diss, K_eq, D_ox, K_ox, L_ox, D_m, K_s_m):
    """
    Solve coupled system with direct parameter input.
    Includes rate-limiting analysis.
    
    Parameters
    ----------
    P_up : float
        Upstream H2 pressure [Pa]
    P_down : float
        Downstream H2 pressure [Pa]
    L_m : float
        Metal thickness [m]
    k_diss : float
        Dissociation rate constant [mol/m²/s/Pa]
    K_eq : float
        Surface equilibrium constant [Pa⁻¹]
    D_ox : float
        Oxide diffusivity [m²/s]
    K_ox : float
        Oxide solubility [mol/m³/Pa^0.5]
    L_ox : float
        Oxide thickness [m]
    D_m : float
        Metal diffusivity [m²/s]
    K_s_m : float
        Metal Sieverts constant [mol/m³/Pa^0.5]
    
    Returns
    -------
    dict
        Contains theta, P_int, J_ss, resistances, and rate-limiting info
    """
    # Compute permeances
    alpha = D_ox * K_ox / L_ox  # Oxide permeance
    beta = D_m * K_s_m / L_m    # Metal permeance
    
    # Solve for θ using brentq
    theta_ss = brentq(
        surface_flux_residual,
        1e-10, 1.0 - 1e-10,
        args=(P_up, alpha, beta, P_down, k_diss, K_eq)
    )
    
    # Calculate √P_int from θ
    sqrt_P_int = sqrt_P_int_from_theta(theta_ss, alpha, beta, K_eq, P_down)
    P_int = sqrt_P_int**2
    
    # Calculate steady-state flux
    J_ss = metal_flux(theta_ss, alpha, beta, K_eq, P_down)
    
    # Verify fluxes
    J_surf = surface_flux(theta_ss, P_up, k_diss, K_eq)
    J_ox = oxide_flux(theta_ss, alpha, beta, K_eq, P_down)
    
    # Surface resistance (linearized approximation)
    R_surface = 1.0 / (k_diss * P_up * (1 - theta_ss)**2) if theta_ss < 0.9 else 0.0
    
    # Oxide resistance
    R_oxide = 1.0 / alpha
    
    # Metal resistance
    R_metal = 1.0 / beta
    
    # Total resistance
    R_total = R_surface + R_oxide + R_metal
    
    # Fractional contributions
    fraction_surface = R_surface / R_total if R_total > 0 else 0
    fraction_oxide = R_oxide / R_total if R_total > 0 else 0
    fraction_metal = R_metal / R_total if R_total > 0 else 0
    
    # Determine rate-limiting step
    if fraction_surface > 0.5:
        rate_limiting = 'surface'
    elif fraction_oxide > 0.5:
        rate_limiting = 'oxide'
    elif fraction_metal > 0.5:
        rate_limiting = 'metal'
    else:
        rate_limiting = 'mixed'
 
    return {
        'theta': theta_ss,
        'P_int': P_int,
        'J_ss': J_ss,
        'J_surface': J_surf,
        'J_oxide': J_ox,
        'J_metal': J_ss,
        'alpha': alpha,
        'beta': beta,
        'rate_limiting': rate_limiting,
        'resistances': {
            'R_surface': R_surface,
            'R_oxide': R_oxide,
            'R_metal': R_metal,
            'R_total': R_total,
            'fraction_surface': fraction_surface,
            'fraction_oxide': fraction_oxide,
            'fraction_metal': fraction_metal,
        },
    }

## 4. Interactive Widget for L2+L6 Model

This widget allows exploration of the perfect oxide + perfect metal system with surface kinetics.

In [4]:
@interact(
    # Operating conditions
    P_down=widgets.FloatLogSlider(value=1e-10, base=10, min=-2, max=4, step=0.5, description='P_down (Pa)'),
    L_m=widgets.FloatLogSlider(value=1e-3, base=10, min=-4, max=-1, step=0.5, description='L_m (m)'),
    # Surface kinetics
    k_diss=widgets.FloatLogSlider(value=1e-15, base=10, min=-18, max=-3, step=0.5, description='k_diss'),
    K_eq=widgets.FloatLogSlider(value=1e-10, base=10, min=-15, max=-1, step=0.5, description='K_eq'),
    # Oxide properties
    D_ox=widgets.FloatLogSlider(value=1e-11, base=10, min=-18, max=-5, step=0.5, description='D_ox (m²/s)'),
    K_ox=widgets.FloatLogSlider(value=1e-6, base=10, min=-14, max=-4, step=0.5, description='K_ox'),
    L_ox=widgets.FloatLogSlider(value=1e-6, base=10, min=-8, max=-4, step=0.5, description='L_ox (m)'),
    # Metal properties
    D_m=widgets.FloatLogSlider(value=1.0e-12, base=10, min=-13, max=-6, step=0.5, description='D_m (m²/s)'),
    K_s_m=widgets.FloatLogSlider(value=3.16e-4, base=10, min=-6, max=0, step=0.5, description='K_s_m'),
)
def interactive_L26_solver(P_down, L_m, k_diss, K_eq,
                           D_ox, K_ox, L_ox, D_m, K_s_m):
    """
    L2+L6 solver with single-pass loop: plot arrays and DataFrame rows
    built together from the same computed values.
    """

    P_up_range = np.logspace(0, 10, 40)  # 1 Pa to 10 GPa

    # Single-pass loop — builds plot arrays and DataFrame rows together
    J_system = []
    theta_values = []
    fraction_surface_list = []
    fraction_oxide_list = []
    fraction_metal_list = []
    rows = []

    for P_up in P_up_range:
        try:
            r = solve_steady_state_flux_direct(
                P_up, P_down, L_m, k_diss, K_eq,
                D_ox, K_ox, L_ox, D_m, K_s_m
            )
            
            # Extract values once
            J_ss = r['J_ss']
            theta = r['theta']
            f_s = r['resistances']['fraction_surface']
            f_o = r['resistances']['fraction_oxide']
            f_m = r['resistances']['fraction_metal']
            
            # Derive rate-limiting label ONCE
            if   f_s > 0.5: rate_lim = 'surface'
            elif f_o > 0.5: rate_lim = 'oxide'
            elif f_m > 0.5: rate_lim = 'metal'
            else:           rate_lim = 'mixed'
            
            # Append to plot arrays
            J_system.append(J_ss)
            theta_values.append(theta)
            fraction_surface_list.append(f_s)
            fraction_oxide_list.append(f_o)
            fraction_metal_list.append(f_m)
            
            # Append to DataFrame rows
            rows.append({
                "P_up (Pa)":           P_up,
                "P_int (Pa)":          r["P_int"],
                "J_ss (mol/m²/s)":     J_ss,
                "θ_surface":           theta,
                "fraction_surface (%)": f_s * 100,
                "fraction_oxide (%)":   f_o * 100,
                "fraction_metal (%)":   f_m * 100,
                "Rate-Limiting":       rate_lim.upper(),
                "α_oxide":             r["alpha"],
                "β_metal":             r["beta"],
            })
            
        except Exception as e:
            J_system.append(np.nan)
            theta_values.append(np.nan)
            fraction_surface_list.append(np.nan)
            fraction_oxide_list.append(np.nan)
            fraction_metal_list.append(np.nan)
            rows.append({"P_up (Pa)": P_up, "Rate-Limiting": "ERROR", "Error": str(e)})

    # Convert to arrays
    J_system = np.array(J_system)
    fraction_surface = np.array(fraction_surface_list)
    fraction_oxide = np.array(fraction_oxide_list)
    fraction_metal = np.array(fraction_metal_list)

    # Rate-limiting array for plot shading
    rate_limiting_arr = np.where(
        fraction_surface > 0.5, 'surface',
        np.where(fraction_oxide > 0.5, 'oxide',
        np.where(fraction_metal > 0.5, 'metal', 'mixed'))
    )

    # Create figure with 2 subplots
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Plot 1: Flux vs Pressure
    valid_idx = ~np.isnan(J_system)
    ax1.loglog(P_up_range, J_system, 'k-', linewidth=2.5, label='L2+L6 Model')

    if np.any(valid_idx):
        P_ref1 = P_up_range[0]
        J_ref1 = J_system[valid_idx][0]
        J_slope1 = J_ref1 * (P_up_range / P_ref1) ** 1.0
        ax1.loglog(P_up_range, J_slope1, 'r--', linewidth=1.5, alpha=0.5, label='Slope = 1 (surface)')

        P_ref05 = P_up_range[-1]
        J_ref05 = J_system[valid_idx][-1]
        J_slope05 = J_ref05 * (P_up_range / P_ref05) ** 0.5
        ax1.loglog(P_up_range, J_slope05, 'g--', linewidth=1.5, alpha=0.5, label='Slope = 0.5 (diffusion)')

    # Rate-limiting regions
    regions = [
        {'mask': rate_limiting_arr == 'surface', 'color': 'red',    'label': 'Surface-limited'},
        {'mask': rate_limiting_arr == 'oxide',   'color': 'orange', 'label': 'Oxide-limited'},
        {'mask': rate_limiting_arr == 'metal',   'color': 'blue',   'label': 'Metal-limited'},
        {'mask': rate_limiting_arr == 'mixed',   'color': 'green',  'label': 'Mixed'},
    ]

    for region in regions:
        mask = region['mask'] & valid_idx
        if np.any(mask):
            idxs = np.where(mask)[0]
            for k, g in groupby(enumerate(idxs), lambda x: x[0] - x[1]):
                group = list(map(itemgetter(1), g))
                if len(group) > 2:
                    P_seg = P_up_range[group]
                    J_seg = J_system[group]
                    ax1.loglog(P_seg, J_seg, color=region['color'], linewidth=4, alpha=0.7)
                    logP = np.log10(P_seg)
                    logJ = np.log10(np.abs(J_seg))
                    slope, _ = np.polyfit(logP, logJ, 1)
                    mid = len(group) // 2
                    ax1.text(P_seg[mid], J_seg[mid], f"{region['label']}\nSlope={slope:.2f}",
                             color=region['color'], fontsize=10, fontweight='bold',
                             bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))
                    
    ax1.set_xlabel('Upstream Pressure $P_{up}$ (Pa)', fontsize=12)
    ax1.set_ylabel('Steady-State Flux $J_{ss}$ (mol/m²/s)', fontsize=12)
    ax1.set_title('L2+L6: Flux vs Pressure', fontsize=14)
    ax1.grid(True, which='both', alpha=0.3)
    ax1.legend(loc='upper left')

    # Plot 2: Rate-Limiting Fractions
    ax2.semilogx(P_up_range, fraction_surface * 100, 'r-', linewidth=2, label='Surface (dissociation)')
    ax2.semilogx(P_up_range, fraction_oxide * 100, 'orange', linewidth=2, label='Oxide (diffusion)')
    ax2.semilogx(P_up_range, fraction_metal * 100, 'b-', linewidth=2, label='Metal (diffusion)')
    ax2.axhline(50, color='gray', linestyle='--', alpha=0.5, label='50% threshold')
    
    ax2.set_xlabel('Upstream Pressure $P_{up}$ (Pa)', fontsize=12)
    ax2.set_ylabel('Resistance Fraction (%)', fontsize=12)
    ax2.set_title('Rate-Limiting Step Analysis', fontsize=14)
    ax2.set_ylim(0, 100)
    ax2.grid(True, alpha=0.3)
    ax2.legend(loc='best')

    plt.tight_layout()
    plt.show()

    # Display DataFrame
    df = pd.DataFrame(rows)
    display(df)

interactive(children=(FloatLogSlider(value=0.01, description='P_down (Pa)', min=-2.0, step=0.5), FloatLogSlide…

## 5. Implement Perfect Oxide + Defective Metal (L4+L6)

This model combines:
- **L6**: Surface kinetics at gas-oxide interface (θ-based)
- **L2**: Oxide diffusion (atomic H, √P dependence)
- **L4**: Defective metal diffusion (GB enhancement + trapping)

The key challenge is that D_eff and θ are coupled and must be solved self-consistently via iteration.

In [5]:
def calculate_defective_metal_flux_L6(
    P_up, P_down, thickness, temperature,
    k_diss, K_eq,
    D_ox, K_ox, L_ox,
    D_lattice, K_s_m,
    microstructure_params,
    lattice_density=1.06e29,
    method='average', n_points=50, mode='both',
    max_iterations=15, tolerance=1e-6
):
    """
    Calculate hydrogen permeation flux through defective metal with surface chemistry.
    
    Combines L6 (surface kinetics), L2 (oxide diffusion), and L4 (defective metal).
    
    Parameters
    ----------
    P_up : float
        Upstream H2 pressure [Pa]
    P_down : float
        Downstream H2 pressure [Pa]
    thickness : float
        Metal thickness [m]
    temperature : float
        Temperature [K]
    k_diss : float
        Dissociation rate constant [mol/m²/s/Pa]
    K_eq : float
        Surface equilibrium constant [Pa⁻¹]
    D_ox : float
        Oxide diffusivity [m²/s]
    K_ox : float
        Oxide solubility [mol/m³/Pa^0.5]
    L_ox : float
        Oxide thickness [m]
    D_lattice : float
        Metal lattice diffusivity [m²/s]
    K_s_m : float
        Metal solubility [mol/m³/Pa^0.5]
    microstructure_params : dict
        Microstructure specification
    lattice_density : float
        Lattice site density [sites/m³]
    method : str
        Method for averaging D_eff: 'average', 'harmonic', 'inlet', 'outlet'
    n_points : int
        Number of points for concentration profile
    mode : str
        Microstructure mode: 'both', 'gb_only', 'trapping_only', 'none'
    max_iterations : int
        Maximum iterations for convergence
    tolerance : float
        Relative tolerance for D_eff convergence
    
    Returns
    -------
    dict
        Flux, concentrations, θ, P_int, D_eff, resistances, etc.
    """
    from calculations.defective_metal import combined_microstructure_model
    
    # Input Validation
    if P_up < 0 or P_down < 0:
        raise ValueError("Pressures must be non-negative")
    if thickness <= 0:
        raise ValueError(f"Thickness must be positive: {thickness} m")
    if D_lattice <= 0:
        raise ValueError("Diffusion coefficient must be positive")
    if k_diss <= 0 or K_eq <= 0:
        raise ValueError("Surface kinetics parameters must be positive")
    
    # Step 1: Compute oxide permeance (fixed)
    alpha = D_ox * K_ox / L_ox
    
    # Step 2: Iterate to find self-consistent D_eff and theta_ss
    D_eff = D_lattice  # Initial guess
    sqrt_P_down = np.sqrt(max(P_down, 0))
    convergence_history = []
    
    for iteration in range(max_iterations):
        beta = D_eff * K_s_m / thickness
        
        # Solve for θ with current beta
        try:
            theta_ss = brentq(
                surface_flux_residual,
                1e-10, 1.0 - 1e-10,
                args=(P_up, alpha, beta, P_down, k_diss, K_eq)
            )
        except ValueError as e:
            return {
                'flux': np.nan,
                'error': f'Failed to solve for theta at iteration {iteration}: {str(e)}',
                'convergence_history': convergence_history
            }
        
        # Calculate sqrt_P_int consistent with this theta and beta
        sqrt_P_int = sqrt_P_int_from_theta(theta_ss, alpha, beta, K_eq, P_down)
        
        # Build concentration profile through metal
        x_array = np.linspace(0, thickness, n_points)
        sqrt_P_array = sqrt_P_int - (sqrt_P_int - sqrt_P_down) * x_array / thickness
        C_array = K_s_m * sqrt_P_array
        
        # Calculate D_eff at each position using microstructure model
        D_array = np.zeros(n_points)
        theta_trap_array = np.zeros(n_points)
        gb_factor_array = np.zeros(n_points)
        
        for i, C_local in enumerate(C_array):
            C_local = max(C_local, 1e-20)
            
            result_i = combined_microstructure_model(
                D_lattice=D_lattice,
                temperature=temperature,
                microstructure_params=microstructure_params,
                lattice_concentration=C_local,
                lattice_density=lattice_density,
                mode=mode
            )
            
            D_array[i] = result_i['D_eff']
            
            if 'trapping' in result_i and result_i['trapping'] is not None:
                theta_trap_array[i] = result_i['trapping'].get('theta_total', 0.0)
            elif 'theta_total' in result_i:
                theta_trap_array[i] = result_i['theta_total']
            else:
                theta_trap_array[i] = 0.0
                
            if 'gb_enhancement' in result_i and result_i['gb_enhancement'] is not None:
                gb_factor_array[i] = result_i['gb_enhancement'].get('factor', 1.0)
            elif 'gb_enhancement_factor' in result_i:
                gb_factor_array[i] = result_i['gb_enhancement_factor']
            else:
                gb_factor_array[i] = 1.0
        
        # Calculate new D_eff based on method
        if method == 'average':
            D_eff_new = np.mean(D_array)
        elif method == 'harmonic':
            D_eff_new = len(D_array) / np.sum(1.0 / D_array)
        elif method == 'inlet':
            D_eff_new = D_array[0]
        elif method == 'outlet':
            D_eff_new = D_array[-1]
        else:
            D_eff_new = np.mean(D_array)
        
        # Track convergence
        convergence_history.append({
            'iteration': iteration,
            'D_eff': D_eff_new,
            'theta': theta_ss,
            'relative_change': abs(D_eff_new - D_eff) / D_eff if D_eff > 0 else np.inf
        })
        
        # Check convergence
        if abs(D_eff_new - D_eff) / D_eff < tolerance:
            D_eff = D_eff_new
            break
        
        D_eff = D_eff_new
    
    # Step 3: Final calculations with converged D_eff and theta_ss
    beta_eff = D_eff * K_s_m / thickness
    sqrt_P_int = sqrt_P_int_from_theta(theta_ss, alpha, beta_eff, K_eq, P_down)
    P_int = sqrt_P_int**2
    
    C_up = K_s_m * sqrt_P_int
    C_down = K_s_m * sqrt_P_down if P_down > 0 else 0.0
    
    # Step 4: Calculate flux through metal
    J_metal = metal_flux(theta_ss, alpha, beta_eff, K_eq, P_down)
    J_surface = surface_flux(theta_ss, P_up, k_diss, K_eq)
    J_oxide = oxide_flux(theta_ss, alpha, beta_eff, K_eq, P_down)
    
    # Step 5: Diagnostic information
    modification_factor = D_eff / D_lattice
    avg_gb_factor = np.mean(gb_factor_array)
    avg_theta_trap = np.mean(theta_trap_array)
    trap_reduction = 1.0 / (1.0 + avg_theta_trap) if avg_theta_trap > 0 else 1.0
    
    if avg_gb_factor > 1.5 and trap_reduction > 0.5:
        dominant_effect = 'gb_enhancement'
    elif avg_gb_factor < 1.5 and trap_reduction < 0.5:
        dominant_effect = 'trapping'
    else:
        dominant_effect = 'balanced'

    # Step 6: Rate-Limiting Analysis
    R_surface_approx = 1.0 / (k_diss * P_up * (1 - theta_ss)**2) if theta_ss < 0.9 else 0.0
    R_oxide = 1.0 / alpha
    R_metal = 1.0 / beta_eff
    R_total = R_surface_approx + R_oxide + R_metal

    fraction_surface = R_surface_approx / R_total if R_total > 0 else 0
    fraction_oxide = R_oxide / R_total if R_total > 0 else 0
    fraction_metal = R_metal / R_total if R_total > 0 else 0

    if fraction_surface > 0.5:
        rate_limiting = 'surface'
    elif fraction_oxide > 0.5:
        rate_limiting = 'oxide'
    elif fraction_metal > 0.5:
        rate_limiting = 'metal'
    else:
        rate_limiting = 'mixed'
    
    return {
        'flux': J_metal,
        'J_surface': J_surface,
        'J_oxide': J_oxide,
        'J_metal': J_metal,
        'theta_surface': theta_ss,
        'P_int': P_int,
        'C_up': C_up,
        'C_down': C_down,
        'alpha': alpha,
        'beta_lattice': D_lattice * K_s_m / thickness,
        'beta_eff': beta_eff,
        'D_eff': D_eff,
        'D_lattice': D_lattice,
        'modification_factor': modification_factor,
        'microstructure_details': {
            'average_theta_trap': avg_theta_trap,
            'average_gb_enhancement': avg_gb_factor,
            'trap_reduction_factor': trap_reduction,
            'dominant_effect': dominant_effect,
            'method_used': method,
        },
        'flux_balance': {
            'J_surface': J_surface,
            'J_oxide': J_oxide,
            'J_metal': J_metal,
            'balanced': np.allclose(J_surface, J_oxide, rtol=1e-4) and np.allclose(J_oxide, J_metal, rtol=1e-4)
        },
        'profiles': {
            'x': x_array,
            'D': D_array,
            'C': C_array,
            'theta_trap': theta_trap_array,
            'gb_factor': gb_factor_array
        },
        'rate_limiting': rate_limiting,
        'resistances': {
            'R_surface': R_surface_approx,
            'R_oxide': R_oxide,
            'R_metal': R_metal,
            'R_total': R_total,
            'fraction_surface': fraction_surface,
            'fraction_oxide': fraction_oxide,
            'fraction_metal': fraction_metal,
        },
        'convergence': {
            'iterations': len(convergence_history),
            'converged': len(convergence_history) < max_iterations,
            'history': convergence_history
        },
        'units': {
            'flux': 'mol/m²/s',
            'concentration': 'mol/m³',
            'pressure': 'Pa',
            'diffusivity': 'm²/s',
            'theta': 'dimensionless'
        }
    }

## 6. Interactive Widget for L4+L6 Model

This widget explores the perfect oxide + defective metal system with microstructure effects.

In [6]:
@interact(
    # Operating conditions
    P_down=widgets.FloatLogSlider(value=1e-10, base=10, min=-2, max=4, step=0.5, description='P_down (Pa)'),
    thickness=widgets.FloatLogSlider(value=1e-3, base=10, min=-4, max=-1, step=0.5, description='L_m (m)'),
    temperature=widgets.IntSlider(value=973, min=573, max=1273, step=50, description='T (K)'),
    # Surface kinetics
    k_diss=widgets.FloatLogSlider(value=1e-11, base=10, min=-18, max=-3, step=0.5, description='k_diss'),
    K_eq=widgets.FloatLogSlider(value=1e-10, base=10, min=-15, max=-1, step=0.5, description='K_eq'),
    # Oxide properties
    D_ox=widgets.FloatLogSlider(value=1e-9, base=10, min=-18, max=-5, step=0.5, description='D_ox (m²/s)'),
    K_ox=widgets.FloatLogSlider(value=3.16e-6, base=10, min=-14, max=-4, step=0.5, description='K_ox'),
    L_ox=widgets.FloatLogSlider(value=1e-6, base=10, min=-8, max=-4, step=0.5, description='L_ox (m)'),
    # Metal properties
    D_lattice=widgets.FloatLogSlider(value=1.0e-10, base=10, min=-12, max=-6, step=0.5, description='D_lattice'),
    K_s_m=widgets.FloatLogSlider(value=3.16e-2, base=10, min=-4, max=0, step=0.5, description='K_s_m'),
    # Microstructure
    grain_size=widgets.FloatLogSlider(value=31e-6, base=10, min=-6, max=-3, step=0.5, description='Grain (m)'),
    trap_density=widgets.FloatLogSlider(value=3.16e15, base=10, min=12, max=18, step=0.5, description='ρ_trap (m⁻²)'),
)
def interactive_L246_solver(P_down, thickness, temperature, k_diss, K_eq,
                            D_ox, K_ox, L_ox, D_lattice, K_s_m,
                            grain_size, trap_density):
    """
    L2+L4+L6 solver with microstructure effects.
    """
    
    # Build microstructure dict from widget inputs
    microstructure = {
        'grain_size': grain_size,
        'grain_shape': 'equiaxed',
        'gb_type': 'LAGB',
        'trap_list': [
            {'name': 'dislocations', 'density': trap_density, 'binding_energy': 27e3}
        ]
    }
    
    P_up_range = np.logspace(0, 12, 40)

    # Single-pass loop
    J_system = []
    theta_values = []
    fraction_surface_list = []
    fraction_oxide_list = []
    fraction_metal_list = []
    rows = []

    for P_up in P_up_range:
        try:
            r = calculate_defective_metal_flux_L6(
                P_up=P_up, P_down=P_down, thickness=thickness, temperature=temperature,
                k_diss=k_diss, K_eq=K_eq,
                D_ox=D_ox, K_ox=K_ox, L_ox=L_ox,
                D_lattice=D_lattice, K_s_m=K_s_m,
                microstructure_params=microstructure
            )
            
            J_ss = r['flux']
            theta = r['theta_surface']
            f_s = r['resistances']['fraction_surface']
            f_o = r['resistances']['fraction_oxide']
            f_m = r['resistances']['fraction_metal']
            
            if   f_s > 0.5: rate_lim = 'surface'
            elif f_o > 0.5: rate_lim = 'oxide'
            elif f_m > 0.5: rate_lim = 'metal'
            else:           rate_lim = 'mixed'
            
            J_system.append(J_ss)
            theta_values.append(theta)
            fraction_surface_list.append(f_s)
            fraction_oxide_list.append(f_o)
            fraction_metal_list.append(f_m)
            
            rows.append({
                "P_up (Pa)":           P_up,
                "P_int (Pa)":          r["P_int"],
                "J_ss (mol/m²/s)":     J_ss,
                "θ_surface":           theta,
                "D_eff (m²/s)":        r["D_eff"],
                "D_eff/D_lattice":     r["modification_factor"],
                "fraction_surface (%)": f_s * 100,
                "fraction_oxide (%)":   f_o * 100,
                "fraction_metal (%)":   f_m * 100,
                "Rate-Limiting":       rate_lim.upper(),
                "α_oxide":             r["alpha"],
                "β_eff":               r["beta_eff"],
            })
            
        except Exception as e:
            J_system.append(np.nan)
            theta_values.append(np.nan)
            fraction_surface_list.append(np.nan)
            fraction_oxide_list.append(np.nan)
            fraction_metal_list.append(np.nan)
            rows.append({"P_up (Pa)": P_up, "Rate-Limiting": "ERROR", "Error": str(e)})

    J_system = np.array(J_system)
    fraction_surface = np.array(fraction_surface_list)
    fraction_oxide = np.array(fraction_oxide_list)
    fraction_metal = np.array(fraction_metal_list)

    rate_limiting_arr = np.where(
        fraction_surface > 0.5, 'surface',
        np.where(fraction_oxide > 0.5, 'oxide',
        np.where(fraction_metal > 0.5, 'metal', 'mixed'))
    )

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    valid_idx = ~np.isnan(J_system)
    ax1.loglog(P_up_range, J_system, 'k-', linewidth=2.5, label='L2+L4+L6 Model')

    if np.any(valid_idx):
        P_ref1 = P_up_range[0]
        J_ref1 = J_system[valid_idx][0]
        J_slope1 = J_ref1 * (P_up_range / P_ref1) ** 1.0
        ax1.loglog(P_up_range, J_slope1, 'r--', linewidth=1.5, alpha=0.5, label='Slope = 1 (surface)')

        P_ref05 = P_up_range[-1]
        J_ref05 = J_system[valid_idx][-1]
        J_slope05 = J_ref05 * (P_up_range / P_ref05) ** 0.5
        ax1.loglog(P_up_range, J_slope05, 'g--', linewidth=1.5, alpha=0.5, label='Slope = 0.5 (diffusion)')

    regions = [
        {'mask': rate_limiting_arr == 'surface', 'color': 'red',    'label': 'Surface-limited'},
        {'mask': rate_limiting_arr == 'oxide',   'color': 'orange', 'label': 'Oxide-limited'},
        {'mask': rate_limiting_arr == 'metal',   'color': 'blue',   'label': 'Metal-limited'},
        {'mask': rate_limiting_arr == 'mixed',   'color': 'green',  'label': 'Mixed'},
    ]

    for region in regions:
        mask = region['mask'] & valid_idx
        if np.any(mask):
            idxs = np.where(mask)[0]
            for k, g in groupby(enumerate(idxs), lambda x: x[0] - x[1]):
                group = list(map(itemgetter(1), g))
                if len(group) > 2:
                    P_seg = P_up_range[group]
                    J_seg = J_system[group]
                    ax1.loglog(P_seg, J_seg, color=region['color'], linewidth=4, alpha=0.7)
                    logP = np.log10(P_seg)
                    logJ = np.log10(np.abs(J_seg))
                    slope, _ = np.polyfit(logP, logJ, 1)
                    mid = len(group) // 2
                    ax1.text(P_seg[mid], J_seg[mid], f"{region['label']}\nSlope={slope:.2f}",
                             color=region['color'], fontsize=10, fontweight='bold',
                             bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))

    ax1.set_xlabel('Upstream Pressure $P_{up}$ (Pa)', fontsize=12)
    ax1.set_ylabel('Steady-State Flux $J_{ss}$ (mol/m²/s)', fontsize=12)
    ax1.set_title('L2+L4+L6: Flux vs Pressure', fontsize=14)
    ax1.grid(True, which='both', alpha=0.3)
    ax1.legend(loc='upper left')

    ax2.semilogx(P_up_range, fraction_surface * 100, 'r-', linewidth=2, label='Surface (dissociation)')
    ax2.semilogx(P_up_range, fraction_oxide * 100, 'orange', linewidth=2, label='Oxide (diffusion)')
    ax2.semilogx(P_up_range, fraction_metal * 100, 'b-', linewidth=2, label='Metal (diffusion)')
    ax2.axhline(50, color='gray', linestyle='--', alpha=0.5, label='50% threshold')
    
    ax2.set_xlabel('Upstream Pressure $P_{up}$ (Pa)', fontsize=12)
    ax2.set_ylabel('Resistance Fraction (%)', fontsize=12)
    ax2.set_title('Rate-Limiting Step Analysis', fontsize=14)
    ax2.set_ylim(0, 100)
    ax2.grid(True, alpha=0.3)
    ax2.legend(loc='best')

    plt.tight_layout()
    plt.show()

    df = pd.DataFrame(rows)
    display(df)

interactive(children=(FloatLogSlider(value=0.01, description='P_down (Pa)', min=-2.0, step=0.5), FloatLogSlide…

## 7. Implement Defective Oxide + Perfect Metal (L3+L6)

### Parallel Path Model

Based on Strehlow & Savage (1974), the total flux through a defective oxide is:
$$J_{total} = f_{intact} \times J_{intact} + \sum_i (f_{defect,i} \times J_{defect,i})$$

Each path uses the L2+L6 coupled model with modified oxide permeance:
- **Intact**: $\alpha_{intact} = D_{ox} \times K_{ox} / L_{ox}$
- **Pinhole**: $\alpha \to \infty$ (no oxide barrier)
- **Crack**: $\alpha_{crack} = \alpha_{intact} / \gamma$ where $\gamma < 1$
- **Grain Boundary**: $\alpha_{gb} = \delta \times \alpha_{intact}$ where $\delta > 1$

In [7]:
def calculate_path_flux_L6(
    P_up, P_down, L_m,
    k_diss, K_eq,
    alpha,
    D_m, K_s_m,
    path_type='intact',
    k_diss_metal=None,
    K_eq_metal=None,
):
    """
    Calculate steady-state flux through a single path (intact or defect).
    
    For pinhole paths:
    - If k_diss_metal and K_eq_metal are provided: use metal surface kinetics
    - If not provided: assume fast kinetics (Sieverts' law limit)
    
    Parameters
    ----------
    P_up : float
        Upstream H2 pressure [Pa]
    P_down : float
        Downstream H2 pressure [Pa]
    L_m : float
        Metal thickness [m]
    k_diss : float
        Surface dissociation rate constant [mol/m²/s/Pa]
    K_eq : float
        Surface equilibrium constant [Pa⁻¹]
    alpha : float or np.inf
        Oxide permeance for this path [mol/m²/s/Pa^0.5]
    D_m : float
        Metal diffusivity [m²/s]
    K_s_m : float
        Metal Sieverts constant [mol/m³/Pa^0.5]
    path_type : str
        'intact', 'pinhole', 'crack', or 'grain_boundary'
    k_diss_metal : float, optional
        Metal surface dissociation rate [mol/m²/s/Pa]. For pinhole only.
    K_eq_metal : float, optional
        Metal surface equilibrium constant [Pa⁻¹]. For pinhole only.
    
    Returns
    -------
    dict
        Contains flux, theta, P_int, resistances, and rate-limiting info
    """
    beta = D_m * K_s_m / L_m
    sqrt_P_down = np.sqrt(max(P_down, 0))
    sqrt_P_up = np.sqrt(max(P_up, 0))
    
    is_pinhole = (path_type == 'pinhole' or alpha == np.inf or alpha > 1e10)
    
    # PINHOLE PATH
    if is_pinhole:
        if k_diss_metal is not None and K_eq_metal is not None:
            k_diss_eff = k_diss_metal
            K_eq_eff = K_eq_metal
            use_sieverts_limit = False
        else:
            use_sieverts_limit = True
        
        if use_sieverts_limit:
            sqrt_P_int = sqrt_P_up
            P_int = P_up
            J_ss = beta * (sqrt_P_int - sqrt_P_down)
            theta_ss = 0.0
            J_surf = J_ss
            J_ox = np.nan
            R_surface = 0.0
            R_oxide = 0.0
            R_metal = 1.0 / beta
            R_total = R_metal
            fraction_surface = 0.0
            fraction_oxide = 0.0
            fraction_metal = 1.0
            rate_limiting = 'metal'
        else:
            def residual(theta):
                if theta <= 0 or theta >= 1:
                    return np.inf
                J_surf = surface_flux(theta, P_up, k_diss_eff, K_eq_eff)
                sqrt_P_int = g_theta(theta, K_eq_eff)
                J_metal = beta * (sqrt_P_int - sqrt_P_down)
                return J_surf - J_metal
            
            try:
                theta_ss = brentq(residual, 1e-10, 1.0 - 1e-10)
            except ValueError as e:
                return {
                    'flux': np.nan, 'theta': np.nan, 'P_int': np.nan,
                    'path_type': path_type, 'alpha': np.inf, 'beta': beta,
                    'error': str(e)
                }
            
            sqrt_P_int = g_theta(theta_ss, K_eq_eff)
            P_int = sqrt_P_int**2
            J_ss = beta * (sqrt_P_int - sqrt_P_down)
            J_surf = surface_flux(theta_ss, P_up, k_diss_eff, K_eq_eff)
            J_ox = np.nan
            
            R_surface = 1.0 / (k_diss_eff * P_up * (1 - theta_ss)**2) if theta_ss < 0.9 else 0.0
            R_oxide = 0.0
            R_metal = 1.0 / beta
            R_total = R_surface + R_metal
            
            fraction_surface = R_surface / R_total if R_total > 0 else 0
            fraction_oxide = 0.0
            fraction_metal = R_metal / R_total if R_total > 0 else 0
            
            if fraction_surface > 0.5:
                rate_limiting = 'surface'
            elif fraction_metal > 0.5:
                rate_limiting = 'metal'
            else:
                rate_limiting = 'mixed'
        
        return {
            'flux': J_ss,
            'theta': theta_ss,
            'P_int': P_int,
            'path_type': path_type,
            'alpha': np.inf,
            'beta': beta,
            'kinetics_used': 'sieverts' if use_sieverts_limit else 'metal',
            'flux_balance': {
                'J_surface': J_surf,
                'J_oxide': J_ox,
                'J_metal': J_ss,
                'balanced': True
            },
            'resistances': {
                'R_surface': R_surface,
                'R_oxide': R_oxide,
                'R_metal': R_metal,
                'R_total': R_total,
                'fraction_surface': fraction_surface,
                'fraction_oxide': fraction_oxide,
                'fraction_metal': fraction_metal,
            },
            'rate_limiting': rate_limiting
        }
    
    # NON-PINHOLE PATHS (intact, crack, GB)
    def residual(theta):
        if theta <= 0 or theta >= 1:
            return np.inf
        J_surf = surface_flux(theta, P_up, k_diss, K_eq)
        sqrt_P_int = sqrt_P_int_from_theta(theta, alpha, beta, K_eq, P_down)
        J_metal = beta * (sqrt_P_int - sqrt_P_down)
        return J_surf - J_metal
    
    try:
        theta_ss = brentq(residual, 1e-10, 1.0 - 1e-10)
    except ValueError as e:
        return {
            'flux': np.nan, 'theta': np.nan, 'P_int': np.nan,
            'path_type': path_type, 'alpha': alpha, 'beta': beta,
            'error': str(e)
        }
    
    sqrt_P_int = sqrt_P_int_from_theta(theta_ss, alpha, beta, K_eq, P_down)
    P_int = sqrt_P_int**2
    J_ss = beta * (sqrt_P_int - sqrt_P_down)
    J_surf = surface_flux(theta_ss, P_up, k_diss, K_eq)
    J_ox = oxide_flux(theta_ss, alpha, beta, K_eq, P_down)
    
    R_surface = 1.0 / (k_diss * P_up * (1 - theta_ss)**2) if theta_ss < 0.9 else 0.0
    R_oxide = 1.0 / alpha
    R_metal = 1.0 / beta
    R_total = R_surface + R_oxide + R_metal
    
    fraction_surface = R_surface / R_total if R_total > 0 else 0
    fraction_oxide = R_oxide / R_total if R_total > 0 else 0
    fraction_metal = R_metal / R_total if R_total > 0 else 0
    
    if fraction_surface > 0.5:
        rate_limiting = 'surface'
    elif fraction_oxide > 0.5:
        rate_limiting = 'oxide'
    elif fraction_metal > 0.5:
        rate_limiting = 'metal'
    else:
        rate_limiting = 'mixed'
    
    return {
        'flux': J_ss,
        'theta': theta_ss,
        'P_int': P_int,
        'path_type': path_type,
        'alpha': alpha,
        'beta': beta,
        'kinetics_used': 'oxide',
        'flux_balance': {
            'J_surface': J_surf,
            'J_oxide': J_ox,
            'J_metal': J_ss,
            'balanced': np.allclose(J_surf, J_ox, rtol=1e-6) and np.allclose(J_ox, J_ss, rtol=1e-6)
        },
        'resistances': {
            'R_surface': R_surface,
            'R_oxide': R_oxide,
            'R_metal': R_metal,
            'R_total': R_total,
            'fraction_surface': fraction_surface,
            'fraction_oxide': fraction_oxide,
            'fraction_metal': fraction_metal,
        },
        'rate_limiting': rate_limiting
    }

## 8. Add Mixed Defect Flux Calculator

This function handles multiple defect types simultaneously, computing area-weighted total flux.

In [8]:
def calculate_mixed_defect_flux_L6(
    P_up, P_down, L_m,
    k_diss, K_eq,
    D_ox, K_ox, L_ox,
    D_m, K_s_m,
    defect_config,
    k_diss_metal=None,
    K_eq_metal=None,
):
    """
    Calculate flux through oxide with multiple defect types.
    
    Parameters
    ----------
    P_up, P_down : float
        Upstream and downstream H2 pressures [Pa]
    L_m : float
        Metal thickness [m]
    k_diss, K_eq : float
        Surface kinetics parameters (oxide surface)
    D_ox, K_ox, L_ox : float
        Intact oxide properties
    D_m, K_s_m : float
        Metal properties
    defect_config : dict
        Configuration for each defect type. Example:
        {
            'pinhole': {'area_fraction': 0.001},
            'crack': {'area_fraction': 0.005, 'thickness_factor': 0.1},
            'grain_boundary': {'area_fraction': 0.02, 'diffusivity_factor': 100}
        }
    k_diss_metal : float, optional
        Metal surface dissociation rate [mol/m²/s/Pa]. For pinhole only.
    K_eq_metal : float, optional
        Metal surface equilibrium constant [Pa⁻¹]. For pinhole only.
    
    Returns
    -------
    dict
        Total flux, individual path contributions, system analysis
    """
    valid_defect_types = ['pinhole', 'crack', 'grain_boundary']
    
    total_defect_fraction = 0.0
    for defect_type, config in defect_config.items():
        if defect_type not in valid_defect_types:
            raise ValueError(f"Unknown defect type: {defect_type}. Valid types: {valid_defect_types}")
        total_defect_fraction += config.get('area_fraction', 0.0)
    
    if total_defect_fraction > 1.0:
        raise ValueError(f"Total defect area fraction ({total_defect_fraction:.3f}) exceeds 1.0")
    
    fraction_intact = 1.0 - total_defect_fraction
    alpha_intact = D_ox * K_ox / L_ox
    
    # Calculate flux through intact path
    try:
        intact_result = calculate_path_flux_L6(
            P_up, P_down, L_m,
            k_diss, K_eq,
            alpha_intact,
            D_m, K_s_m,
            path_type='intact'
        )
        
        if 'error' in intact_result:
            return {
                'J_total': np.nan,
                'error': f"Intact path failed: {intact_result['error']}",
            }
    except Exception as e:
        return {
            'J_total': np.nan,
            'error': f"Intact path calculation failed: {e}",
        }
    
    # Calculate flux through each defect path
    defect_results = {}
    
    for defect_type, config in defect_config.items():
        fraction_defect = config.get('area_fraction', 0.0)
        
        if fraction_defect <= 0:
            continue
        
        if defect_type == 'pinhole':
            alpha_defect = np.inf
        elif defect_type == 'crack':
            gamma = config.get('thickness_factor', 0.1)
            alpha_defect = alpha_intact / gamma
        elif defect_type == 'grain_boundary':
            delta = config.get('diffusivity_factor', 100.0)
            alpha_defect = delta * alpha_intact
        else:
            continue
        
        try:
            path_result = calculate_path_flux_L6(
                P_up, P_down, L_m,
                k_diss, K_eq,
                alpha_defect,
                D_m, K_s_m,
                path_type=defect_type,
                k_diss_metal=k_diss_metal if defect_type == 'pinhole' else None,
                K_eq_metal=K_eq_metal if defect_type == 'pinhole' else None
            )
            
            if 'error' in path_result:
                print(f"Warning: {defect_type} path failed: {path_result['error']}")
                continue
                
        except Exception as e:
            print(f"Warning: {defect_type} path calculation failed: {e}")
            continue
        
        defect_results[defect_type] = {
            'area_fraction': fraction_defect,
            'alpha': alpha_defect,
            'alpha_ratio': alpha_defect / alpha_intact if alpha_defect != np.inf else np.inf,
            'path_result': path_result,
            'flux_contribution': fraction_defect * path_result['flux'],
        }
    
    # Calculate total flux (area-weighted sum)
    J_intact_contribution = fraction_intact * intact_result['flux']
    J_total = J_intact_contribution
    
    for defect_type, data in defect_results.items():
        J_total += data['flux_contribution']
    
    # Flux breakdown analysis
    flux_breakdown = {
        'intact': {
            'area_fraction': fraction_intact,
            'flux': intact_result['flux'],
            'contribution': J_intact_contribution,
            'fraction_of_total': J_intact_contribution / J_total if J_total > 0 else 0,
        }
    }
    
    for defect_type, data in defect_results.items():
        flux_breakdown[defect_type] = {
            'area_fraction': data['area_fraction'],
            'flux': data['path_result']['flux'],
            'contribution': data['flux_contribution'],
            'fraction_of_total': data['flux_contribution'] / J_total if J_total > 0 else 0,
        }
    
    # Determine dominant path
    max_contribution = J_intact_contribution
    dominant_path = 'intact'
    
    for defect_type, data in defect_results.items():
        if data['flux_contribution'] > max_contribution:
            max_contribution = data['flux_contribution']
            dominant_path = defect_type
    
    dominant_fraction = max_contribution / J_total if J_total > 0 else 0
    if dominant_fraction < 0.7:
        dominant_path = 'mixed'
    
    # System-level rate-limiting analysis
    if dominant_path == 'intact':
        dominant_rate_limiting = intact_result['rate_limiting']
        dominant_resistances = intact_result['resistances']
    elif dominant_path in defect_results:
        dominant_rate_limiting = defect_results[dominant_path]['path_result']['rate_limiting']
        dominant_resistances = defect_results[dominant_path]['path_result']['resistances']
    else:
        dominant_rate_limiting = 'mixed (multiple paths)'
        dominant_resistances = None
    
    enhancement_factor = J_total / intact_result['flux'] if intact_result['flux'] > 0 else np.inf
    
    return {
        'J_total': J_total,
        'enhancement_factor': enhancement_factor,
        'dominant_path': dominant_path,
        'dominant_fraction': dominant_fraction,
        'flux_breakdown': flux_breakdown,
        'fraction_intact': fraction_intact,
        'total_defect_fraction': total_defect_fraction,
        'intact_path': intact_result,
        'defect_paths': defect_results,
        'system_rate_limiting': dominant_rate_limiting,
        'dominant_resistances': dominant_resistances,
        'alpha_intact': alpha_intact,
        'defect_config': defect_config,
        'units': {
            'flux': 'mol/m²/s',
            'pressure': 'Pa',
            'permeance': 'mol/m²/s/Pa^0.5',
            'area_fraction': 'dimensionless',
        }
    }

## 9. Interactive Widget for L3+L6 Model

This widget explores the defective oxide + perfect metal system with parallel paths.

In [9]:
@interact(
    # Operating conditions
    P_down=widgets.FloatLogSlider(value=1e0, base=10, min=-2, max=4, step=0.5, description='P_down (Pa)'),
    L_m=widgets.FloatLogSlider(value=1e-3, base=10, min=-4, max=-1, step=0.5, description='L_m (m)'),
    # Surface kinetics
    k_diss=widgets.FloatLogSlider(value=1e-18, base=10, min=-18, max=-10, step=0.5, description='k_diss'),
    K_eq=widgets.FloatLogSlider(value=1e-5, base=10, min=-15, max=-5, step=0.5, description='K_eq'),
    # Metal surface kinetics for pinhole
    k_diss_metal=widgets.FloatLogSlider(value=1e-12, base=10, min=-15, max=-8, step=0.5, description='k_diss_metal'),
    K_eq_metal=widgets.FloatLogSlider(value=1e-8, base=10, min=-12, max=-4, step=0.5, description='K_eq_metal'),
    use_sieverts_pinhole=widgets.Checkbox(value=True, description='Sieverts limit for pinhole'),
    # Oxide properties
    D_ox=widgets.FloatLogSlider(value=1e-11, base=10, min=-22, max=-9, step=0.5, description='D_ox (m²/s)'),
    K_ox=widgets.FloatLogSlider(value=1e-10, base=10, min=-12, max=0, step=0.5, description='K_ox'),
    L_ox=widgets.FloatLogSlider(value=1e-7, base=10, min=-8, max=-4, step=0.5, description='L_ox (m)'),
    # Metal properties
    D_m=widgets.FloatLogSlider(value=1e-12, base=10, min=-12, max=-6, step=0.5, description='D_m (m²/s)'),
    K_s_m=widgets.FloatLogSlider(value=0.000316, base=10, min=-6, max=0, step=0.5, description='K_s_m'),
    # Defect parameters
    fraction_pinhole=widgets.FloatSlider(value=0.1, min=0, max=5, step=0.1, description='Pinhole %'),
    fraction_crack=widgets.FloatSlider(value=0.5, min=0, max=10, step=0.5, description='Crack %'),
    fraction_gb=widgets.FloatSlider(value=0.2, min=0, max=20, step=1.0, description='GB %'),
    gamma=widgets.FloatLogSlider(value=0.1, base=10, min=-2, max=0, step=0.5, description='γ (crack)'),
    delta=widgets.FloatLogSlider(value=100, base=10, min=0, max=4, step=0.5, description='δ (GB)'),
)
def interactive_L36_solver(P_down, L_m, k_diss, K_eq, k_diss_metal, K_eq_metal, use_sieverts_pinhole,
                           D_ox, K_ox, L_ox, D_m, K_s_m,
                           fraction_pinhole, fraction_crack, fraction_gb, gamma, delta):
    """
    Interactive solver for L3+L6 parallel path model.
    """
    
    # Check total defect fraction
    total_defect = fraction_pinhole + fraction_crack + fraction_gb
    if total_defect > 1.0:
        print(f"⚠️ Total defect fraction ({total_defect*100:.1f}%) exceeds 100%!")
        return
    
    # Build defect config
    defect_config = {}
    if fraction_pinhole > 0:
        defect_config['pinhole'] = {'area_fraction': fraction_pinhole}
    if fraction_crack > 0:
        defect_config['crack'] = {'area_fraction': fraction_crack, 'thickness_factor': gamma}
    if fraction_gb > 0:
        defect_config['grain_boundary'] = {'area_fraction': fraction_gb, 'diffusivity_factor': delta}

    P_up_range = np.logspace(2, 12, 40)
    alpha_intact = D_ox * K_ox / L_ox
    
    # Single-pass loop
    plot_data = {k: [] for k in [
        'J_total', 'J_intact', 'enhancement', 'theta_intact',
        'frac_intact', 'frac_pinhole', 'frac_crack', 'frac_gb',
        'fraction_surface', 'fraction_oxide', 'fraction_metal',
    ]}
    rows = []
    
    for P_up in P_up_range:
        try:
            if defect_config:
                r = calculate_mixed_defect_flux_L6(
                    P_up, P_down, L_m, k_diss, K_eq,
                    D_ox, K_ox, L_ox, D_m, K_s_m,
                    defect_config=defect_config,
                    k_diss_metal=None if use_sieverts_pinhole else k_diss_metal,
                    K_eq_metal=None if use_sieverts_pinhole else K_eq_metal,
                )
                
                J_tot = r['J_total']
                J_int = r['intact_path']['flux']
                theta = r['intact_path']['theta']
                P_int = r['intact_path']['P_int']
                enhancement = r['enhancement_factor']
                
                frac_intact = r['flux_breakdown']['intact']['fraction_of_total']
                frac_pinhole = r['flux_breakdown'].get('pinhole', {}).get('fraction_of_total', 0)
                frac_crack = r['flux_breakdown'].get('crack', {}).get('fraction_of_total', 0)
                frac_gb = r['flux_breakdown'].get('grain_boundary', {}).get('fraction_of_total', 0)
                
                # Flux-weighted resistances across ALL paths
                ws = wo = wm = 0.0
                J_intact_contrib = frac_intact * J_tot
                
                res_intact = r['intact_path']['resistances']
                w = J_intact_contrib / J_tot if J_tot > 0 else 0.0
                ws += w * res_intact['fraction_surface']
                wo += w * res_intact['fraction_oxide']
                wm += w * res_intact['fraction_metal']
                
                for dt, data in r['defect_paths'].items():
                    pr = data['path_result']
                    J_c = data['flux_contribution']
                    w = J_c / J_tot if J_tot > 0 else 0.0
                    ws += w * pr['resistances']['fraction_surface']
                    wo += w * pr['resistances']['fraction_oxide']
                    wm += w * pr['resistances']['fraction_metal']
                
                if   ws > 0.5: rate_lim = 'surface'
                elif wo > 0.5: rate_lim = 'oxide'
                elif wm > 0.5: rate_lim = 'metal'
                else:          rate_lim = 'mixed'
                
                plot_data['J_total'].append(J_tot)
                plot_data['J_intact'].append(J_int)
                plot_data['enhancement'].append(enhancement)
                plot_data['theta_intact'].append(theta)
                plot_data['frac_intact'].append(frac_intact)
                plot_data['frac_pinhole'].append(frac_pinhole)
                plot_data['frac_crack'].append(frac_crack)
                plot_data['frac_gb'].append(frac_gb)
                plot_data['fraction_surface'].append(ws)
                plot_data['fraction_oxide'].append(wo)
                plot_data['fraction_metal'].append(wm)
                
                rows.append({
                    "P_up (Pa)":           P_up,
                    "J_total (mol/m²/s)":  J_tot,
                    "J_intact (mol/m²/s)": J_int,
                    "Enhancement":         enhancement,
                    "θ_intact":            theta,
                    "P_int_intact (Pa)":   P_int,
                    "fraction_intact (%)": frac_intact * 100,
                    "fraction_pinhole (%)": frac_pinhole * 100,
                    "fraction_crack (%)":  frac_crack * 100,
                    "fraction_gb (%)":     frac_gb * 100,
                    "Dominant Path":       r["dominant_path"].upper(),
                    "Rate-Limiting":       rate_lim.upper(),
                    "α_intact":            r["alpha_intact"],
                })
                
            else:
                r = calculate_path_flux_L6(
                    P_up, P_down, L_m, k_diss, K_eq,
                    alpha_intact, D_m, K_s_m, path_type='intact'
                )
                
                J_ss = r['flux']
                theta = r['theta']
                P_int = r['P_int']
                ws = r['resistances']['fraction_surface']
                wo = r['resistances']['fraction_oxide']
                wm = r['resistances']['fraction_metal']
                
                if   ws > 0.5: rate_lim = 'surface'
                elif wo > 0.5: rate_lim = 'oxide'
                elif wm > 0.5: rate_lim = 'metal'
                else:          rate_lim = 'mixed'
                
                plot_data['J_total'].append(J_ss)
                plot_data['J_intact'].append(J_ss)
                plot_data['enhancement'].append(1.0)
                plot_data['theta_intact'].append(theta)
                plot_data['frac_intact'].append(1.0)
                plot_data['frac_pinhole'].append(0)
                plot_data['frac_crack'].append(0)
                plot_data['frac_gb'].append(0)
                plot_data['fraction_surface'].append(ws)
                plot_data['fraction_oxide'].append(wo)
                plot_data['fraction_metal'].append(wm)
                
                rows.append({
                    "P_up (Pa)":           P_up,
                    "J_total (mol/m²/s)":  J_ss,
                    "J_intact (mol/m²/s)": J_ss,
                    "Enhancement":         1.0,
                    "θ_intact":            theta,
                    "P_int_intact (Pa)":   P_int,
                    "fraction_intact (%)": 100.0,
                    "fraction_pinhole (%)": 0.0,
                    "fraction_crack (%)":  0.0,
                    "fraction_gb (%)":     0.0,
                    "Dominant Path":       "INTACT",
                    "Rate-Limiting":       rate_lim.upper(),
                    "α_intact":            r["alpha"],
                })
                
        except Exception as e:
            for k in plot_data:
                plot_data[k].append(np.nan)
            rows.append({"P_up (Pa)": P_up, "Rate-Limiting": "ERROR", "Error": str(e)})
    
    # Convert to arrays
    J_total = np.array(plot_data['J_total'])
    J_intact = np.array(plot_data['J_intact'])
    fraction_surface = np.array(plot_data['fraction_surface'])
    fraction_oxide = np.array(plot_data['fraction_oxide'])
    fraction_metal = np.array(plot_data['fraction_metal'])
    
    rate_limiting_arr = np.where(
        fraction_surface > 0.5, 'surface',
        np.where(fraction_oxide > 0.5, 'oxide',
        np.where(fraction_metal > 0.5, 'metal', 'mixed'))
    )
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    valid_idx = ~np.isnan(J_total)
    ax1.loglog(P_up_range, J_total, 'k-', linewidth=2.5, label='L3+L6 Model (Total)')
    ax1.loglog(P_up_range, J_intact, 'b--', linewidth=1.5, alpha=0.7, label='Intact only')
    
    if np.any(valid_idx):
        P_ref1 = P_up_range[0]
        J_ref1 = J_total[valid_idx][0]
        J_slope1 = J_ref1 * (P_up_range / P_ref1) ** 1.0
        ax1.loglog(P_up_range, J_slope1, 'r--', linewidth=1.5, alpha=0.5, label='Slope = 1 (surface)')

        P_ref05 = P_up_range[-1]
        J_ref05 = J_total[valid_idx][-1]
        J_slope05 = J_ref05 * (P_up_range / P_ref05) ** 0.5
        ax1.loglog(P_up_range, J_slope05, 'g--', linewidth=1.5, alpha=0.5, label='Slope = 0.5 (diffusion)')
    
    regions = [
        {'mask': rate_limiting_arr == 'surface', 'color': 'red',    'label': 'Surface-limited'},
        {'mask': rate_limiting_arr == 'oxide',   'color': 'orange', 'label': 'Oxide-limited'},
        {'mask': rate_limiting_arr == 'metal',   'color': 'blue',   'label': 'Metal-limited'},
        {'mask': rate_limiting_arr == 'mixed',   'color': 'green',  'label': 'Mixed'},
    ]

    for region in regions:
        mask = region['mask'] & valid_idx
        if np.any(mask):
            idxs = np.where(mask)[0]
            for k, g in groupby(enumerate(idxs), lambda x: x[0] - x[1]):
                group = list(map(itemgetter(1), g))
                if len(group) > 2:
                    P_seg = P_up_range[group]
                    J_seg = J_total[group]
                    ax1.loglog(P_seg, J_seg, color=region['color'], linewidth=4, alpha=0.7)
                    slope, _ = np.polyfit(np.log10(P_seg), np.log10(np.abs(J_seg)), 1)
                    mid = len(group) // 2
                    ax1.text(P_seg[mid], J_seg[mid], f"{region['label']}\nSlope={slope:.2f}",
                             color=region['color'], fontsize=10, fontweight='bold',
                             bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))

    ax1.set_xlabel('Upstream Pressure $P_{up}$ (Pa)', fontsize=12)
    ax1.set_ylabel('Steady-State Flux $J_{ss}$ (mol/m²/s)', fontsize=12)
    ax1.set_title('L3+L6: Flux vs Pressure (Defective Oxide)', fontsize=14)
    ax1.grid(True, which='both', alpha=0.3)
    ax1.legend(loc='upper left')

    ax2.semilogx(P_up_range, fraction_surface * 100, 'r-', linewidth=2, label='Surface (dissociation)')
    ax2.semilogx(P_up_range, fraction_oxide * 100, 'orange', linewidth=2, label='Oxide (diffusion)')
    ax2.semilogx(P_up_range, fraction_metal * 100, 'b-', linewidth=2, label='Metal (diffusion)')
    ax2.axhline(50, color='gray', linestyle='--', alpha=0.5, label='50% threshold')
    
    ax2.set_xlabel('Upstream Pressure $P_{up}$ (Pa)', fontsize=12)
    ax2.set_ylabel('Resistance Fraction (%)', fontsize=12)
    ax2.set_title('Rate-Limiting Step Analysis', fontsize=14)
    ax2.set_ylim(0, 100)
    ax2.grid(True, alpha=0.3)
    ax2.legend(loc='best')

    plt.tight_layout()
    plt.show()
    
    df = pd.DataFrame(rows)
    display(df)

interactive(children=(FloatLogSlider(value=1.0, description='P_down (Pa)', min=-2.0, step=0.5), FloatLogSlider…

## 10. Implement Full Model (L3+L4+L6)

This combines all three physics layers:
- **L6**: Surface kinetics at gas-oxide interface
- **L3**: Parallel oxide paths (intact + defects)
- **L4**: Defective metal microstructure

Each oxide path now has its own microstructure calculation through the metal.

In [10]:
def calculate_path_flux_L346_v2(
    P_up, P_down, L_m, temperature,
    k_diss, K_eq,
    alpha,
    D_lattice, K_s_m,
    microstructure_params,
    path_type='intact',
    lattice_density=1.06e29,
    method='average',
    n_points=20,
    mode='both',
    max_iterations=10,
    tolerance=1e-5
):
    """
    Calculate flux through a single oxide path with defective metal.
    
    Combines L6 (surface kinetics), variable α (oxide path type), and L4 (microstructure).
    
    Parameters
    ----------
    P_up, P_down : float
        Upstream and downstream H2 pressures [Pa]
    L_m : float
        Metal thickness [m]
    temperature : float
        Temperature [K]
    k_diss : float
        Surface dissociation rate constant [mol/m²/s/Pa]
    K_eq : float
        Surface equilibrium constant [Pa⁻¹]
    alpha : float or np.inf
        Oxide permeance for this path [mol/m²/s/Pa^0.5]
    D_lattice : float
        Metal lattice diffusivity [m²/s]
    K_s_m : float
        Metal Sieverts constant [mol/m³/Pa^0.5]
    microstructure_params : dict
        Metal microstructure specification
    path_type : str
        'intact', 'pinhole', 'crack', or 'grain_boundary'
    
    Returns
    -------
    dict
        Flux, theta, P_int, D_eff, resistances, rate-limiting info
    """
    from calculations.defective_metal import combined_microstructure_model
    
    is_pinhole = (path_type == 'pinhole' or alpha == np.inf or alpha > 1e10)
    sqrt_P_down = np.sqrt(max(P_down, 0))
    
    # Iterative solution: D_eff and θ are coupled
    D_eff = D_lattice
    convergence_history = []
    
    for iteration in range(max_iterations):
        beta = D_eff * K_s_m / L_m
        
        def residual(theta):
            if theta <= 0 or theta >= 1:
                return np.inf
            
            J_surf = surface_flux(theta, P_up, k_diss, K_eq)
            
            if is_pinhole:
                sqrt_P_int = g_theta(theta, K_eq)
            else:
                sqrt_P_int = sqrt_P_int_from_theta(theta, alpha, beta, K_eq, P_down)
            
            J_metal = beta * (sqrt_P_int - sqrt_P_down)
            return J_surf - J_metal
        
        try:
            theta_ss = brentq(residual, 1e-12, 1.0 - 1e-12)
        except ValueError as e:
            return {
                'flux': np.nan,
                'theta': np.nan,
                'P_int': np.nan,
                'D_eff': np.nan,
                'path_type': path_type,
                'error': f'Failed to solve for theta: {str(e)}',
                'iteration': iteration
            }
        
        if is_pinhole:
            sqrt_P_int = g_theta(theta_ss, K_eq)
        else:
            sqrt_P_int = sqrt_P_int_from_theta(theta_ss, alpha, beta, K_eq, P_down)
        
        # Calculate D_eff through metal using microstructure model
        x_array = np.linspace(0, L_m, n_points)
        sqrt_P_array = sqrt_P_int - (sqrt_P_int - sqrt_P_down) * x_array / L_m
        C_array = K_s_m * sqrt_P_array
        
        D_array = np.zeros(n_points)
        theta_trap_array = np.zeros(n_points)
        gb_factor_array = np.zeros(n_points)
        
        for i, C_local in enumerate(C_array):
            C_local = max(C_local, 1e-20)
            
            result_i = combined_microstructure_model(
                D_lattice=D_lattice,
                temperature=temperature,
                microstructure_params=microstructure_params,
                lattice_concentration=C_local,
                lattice_density=lattice_density,
                mode=mode
            )
            
            D_array[i] = result_i['D_eff']
            
            if 'trapping' in result_i and result_i['trapping'] is not None:
                theta_trap_array[i] = result_i['trapping'].get('theta_total', 0.0)
            elif 'theta_total' in result_i:
                theta_trap_array[i] = result_i['theta_total']
            
            if 'gb_enhancement' in result_i and result_i['gb_enhancement'] is not None:
                gb_factor_array[i] = result_i['gb_enhancement'].get('factor', 1.0)
            elif 'gb_enhancement_factor' in result_i:
                gb_factor_array[i] = result_i['gb_enhancement_factor']
            else:
                gb_factor_array[i] = 1.0
        
        if method == 'average':
            D_eff_new = np.mean(D_array)
        elif method == 'harmonic':
            D_eff_new = len(D_array) / np.sum(1.0 / D_array)
        elif method == 'inlet':
            D_eff_new = D_array[0]
        elif method == 'outlet':
            D_eff_new = D_array[-1]
        else:
            D_eff_new = np.mean(D_array)
        
        rel_change = abs(D_eff_new - D_eff) / D_eff if D_eff > 0 else np.inf
        convergence_history.append({
            'iteration': iteration,
            'D_eff': D_eff_new,
            'theta': theta_ss,
            'relative_change': rel_change
        })
        
        if rel_change < tolerance:
            D_eff = D_eff_new
            break
        
        D_eff = D_eff_new
    
    # Final calculations with converged values
    beta_eff = D_eff * K_s_m / L_m
    P_int = sqrt_P_int**2
    
    J_metal = beta_eff * (sqrt_P_int - sqrt_P_down)
    J_surface = surface_flux(theta_ss, P_up, k_diss, K_eq)
    
    if is_pinhole:
        J_oxide = np.nan
        R_oxide = 0.0
        alpha_out = np.inf
    else:
        J_oxide = oxide_flux(theta_ss, alpha, beta_eff, K_eq, P_down)
        R_oxide = 1.0 / alpha
        alpha_out = alpha
    
    R_surface = 1.0 / (k_diss * P_up * (1 - theta_ss)**2) if theta_ss < 0.9 else 0.0
    R_metal = 1.0 / beta_eff
    R_total = R_surface + R_oxide + R_metal
    
    fraction_surface = R_surface / R_total if R_total > 0 else 0
    fraction_oxide = R_oxide / R_total if R_total > 0 else 0
    fraction_metal = R_metal / R_total if R_total > 0 else 0
    
    if fraction_surface > 0.5:
        rate_limiting = 'surface'
    elif fraction_oxide > 0.5:
        rate_limiting = 'oxide'
    elif fraction_metal > 0.5:
        rate_limiting = 'metal'
    else:
        rate_limiting = 'mixed'
    
    modification_factor = D_eff / D_lattice
    avg_gb_factor = np.mean(gb_factor_array)
    avg_theta_trap = np.mean(theta_trap_array)
    
    return {
        'flux': J_metal,
        'theta': theta_ss,
        'P_int': P_int,
        'path_type': path_type,
        'alpha': alpha_out,
        'beta_lattice': D_lattice * K_s_m / L_m,
        'beta_eff': beta_eff,
        'D_eff': D_eff,
        'D_lattice': D_lattice,
        'modification_factor': modification_factor,
        'microstructure': {
            'avg_gb_factor': avg_gb_factor,
            'avg_theta_trap': avg_theta_trap,
        },
        'flux_balance': {
            'J_surface': J_surface,
            'J_oxide': J_oxide,
            'J_metal': J_metal,
        },
        'resistances': {
            'R_surface': R_surface,
            'R_oxide': R_oxide,
            'R_metal': R_metal,
            'R_total': R_total,
            'fraction_surface': fraction_surface,
            'fraction_oxide': fraction_oxide,
            'fraction_metal': fraction_metal,
        },
        'rate_limiting': rate_limiting,
        'convergence': {
            'iterations': len(convergence_history),
            'converged': len(convergence_history) < max_iterations,
            'history': convergence_history
        },
        'profiles': {
            'x': x_array,
            'D': D_array,
            'C': C_array,
            'theta_trap': theta_trap_array,
            'gb_factor': gb_factor_array
        }
    }


def calculate_full_model_flux_L346_v2(
    P_up, P_down, L_m, temperature,
    k_diss, K_eq,
    D_ox, K_ox, L_ox,
    D_lattice, K_s_m,
    microstructure_params,
    defect_config,
    lattice_density=1.06e29,
    method='average',
    n_points=20,
    mode='both'
):
    """
    Calculate total flux with defective oxide AND defective metal.
    
    Full L3 + L4 + L6 model.
    
    Parameters
    ----------
    P_up, P_down : float
        Upstream and downstream pressures [Pa]
    L_m : float
        Metal thickness [m]
    temperature : float
        Temperature [K]
    k_diss, K_eq : float
        Surface kinetics parameters
    D_ox, K_ox, L_ox : float
        Intact oxide properties
    D_lattice, K_s_m : float
        Metal lattice properties
    microstructure_params : dict
        Metal microstructure (grain size, traps, etc.)
    defect_config : dict
        Oxide defect configuration
    
    Returns
    -------
    dict
        Total flux, path contributions, microstructure effects, rate-limiting
    """
    valid_defect_types = ['pinhole', 'crack', 'grain_boundary']
    
    total_defect_fraction = 0.0
    for defect_type, config in defect_config.items():
        if defect_type not in valid_defect_types:
            raise ValueError(f"Unknown defect type: {defect_type}")
        total_defect_fraction += config.get('area_fraction', 0.0)
    
    if total_defect_fraction > 1.0:
        raise ValueError(f"Total defect fraction ({total_defect_fraction:.3f}) exceeds 1.0")
    
    fraction_intact = 1.0 - total_defect_fraction
    alpha_intact = D_ox * K_ox / L_ox
    
    # Calculate flux through intact path
    intact_result = calculate_path_flux_L346_v2(
        P_up, P_down, L_m, temperature,
        k_diss, K_eq,
        alpha_intact,
        D_lattice, K_s_m,
        microstructure_params,
        path_type='intact',
        lattice_density=lattice_density,
        method=method,
        n_points=n_points,
        mode=mode
    )
    
    # Calculate flux through each defect path
    defect_results = {}
    
    for defect_type, config in defect_config.items():
        fraction_defect = config.get('area_fraction', 0.0)
        
        if fraction_defect <= 0:
            continue
        
        if defect_type == 'pinhole':
            alpha_defect = np.inf
        elif defect_type == 'crack':
            gamma = config.get('thickness_factor', 0.1)
            alpha_defect = alpha_intact / gamma
        elif defect_type == 'grain_boundary':
            delta = config.get('diffusivity_factor', 100.0)
            alpha_defect = delta * alpha_intact
        
        path_result = calculate_path_flux_L346_v2(
            P_up, P_down, L_m, temperature,
            k_diss, K_eq,
            alpha_defect,
            D_lattice, K_s_m,
            microstructure_params,
            path_type=defect_type,
            lattice_density=lattice_density,
            method=method,
            n_points=n_points,
            mode=mode
        )
        
        defect_results[defect_type] = {
            'area_fraction': fraction_defect,
            'alpha': alpha_defect,
            'alpha_ratio': alpha_defect / alpha_intact if alpha_defect != np.inf else np.inf,
            'path_result': path_result,
            'flux_contribution': fraction_defect * path_result['flux'],
        }
    
    # Calculate total flux (area-weighted sum)
    J_intact_contribution = fraction_intact * intact_result['flux']
    J_total = J_intact_contribution
    
    for defect_type, data in defect_results.items():
        J_total += data['flux_contribution']
    
    # Flux breakdown analysis
    flux_breakdown = {
        'intact': {
            'area_fraction': fraction_intact,
            'flux': intact_result['flux'],
            'contribution': J_intact_contribution,
            'fraction_of_total': J_intact_contribution / J_total if J_total > 0 else 0,
            'D_eff': intact_result['D_eff'],
            'modification_factor': intact_result['modification_factor'],
            'theta': intact_result['theta'],
            'P_int': intact_result['P_int'],
        }
    }
    
    for defect_type, data in defect_results.items():
        pr = data['path_result']
        flux_breakdown[defect_type] = {
            'area_fraction': data['area_fraction'],
            'flux': pr['flux'],
            'contribution': data['flux_contribution'],
            'fraction_of_total': data['flux_contribution'] / J_total if J_total > 0 else 0,
            'D_eff': pr['D_eff'],
            'modification_factor': pr['modification_factor'],
            'theta': pr['theta'],
            'P_int': pr['P_int'],
        }
    
    # Determine dominant path
    max_contribution = J_intact_contribution
    dominant_path = 'intact'
    
    for defect_type, data in defect_results.items():
        if data['flux_contribution'] > max_contribution:
            max_contribution = data['flux_contribution']
            dominant_path = defect_type
    
    dominant_fraction = max_contribution / J_total if J_total > 0 else 0
    if dominant_fraction < 0.7:
        dominant_path = 'mixed'
    
    enhancement_vs_intact = J_total / intact_result['flux'] if intact_result['flux'] > 0 else np.inf
    
    # System-level rate-limiting
    if dominant_path == 'intact':
        system_rate_limiting = intact_result['rate_limiting']
        system_resistances = intact_result['resistances']
    elif dominant_path in defect_results:
        system_rate_limiting = defect_results[dominant_path]['path_result']['rate_limiting']
        system_resistances = defect_results[dominant_path]['path_result']['resistances']
    else:
        system_rate_limiting = 'mixed'
        system_resistances = None
    
    # Average D_eff across all paths (flux-weighted)
    D_eff_avg = 0.0
    for path, data in flux_breakdown.items():
        D_eff_avg += data['fraction_of_total'] * data['D_eff']
    
    return {
        'J_total': J_total,
        'enhancement_vs_intact': enhancement_vs_intact,
        'dominant_path': dominant_path,
        'dominant_fraction': dominant_fraction,
        'flux_breakdown': flux_breakdown,
        'fraction_intact': fraction_intact,
        'total_defect_fraction': total_defect_fraction,
        'intact_path': intact_result,
        'defect_paths': defect_results,
        'D_eff_avg': D_eff_avg,
        'D_lattice': D_lattice,
        'overall_modification_factor': D_eff_avg / D_lattice,
        'system_rate_limiting': system_rate_limiting,
        'system_resistances': system_resistances,
        'alpha_intact': alpha_intact,
        'defect_config': defect_config,
        'microstructure_params': microstructure_params,
    }

## 11. Interactive Widget for Full L3+L4+L6 Model

This comprehensive widget combines all parameters for the full physics model.